In [1]:
# Step 0: Import the Pandas library for data analysis
import pandas as pd

In [2]:
# Step 1: Load the raw Iowa recidivism CSV into a Pandas DataFrame
df = pd.read_csv("../01_raw_data/iowa_prison_recidivism_status_984_rows.csv")

# Preview the first 5 records
df.head()

,record_id,offender_cd,birth_date,age,Race,sex,region_exit,exit_sup,cohort_fiscal_year,report_fiscal_year,...,submitted_date,total_violence_score,violence_risk,total_victimization_score,victimization_risk,ranking_number,risk_ranking,regcd_exit,work_unit_region_nm,work_unit_nm
0,NaN,NaN,NaN,25,White,Male,Mount Pleasant Correctional Facility,Work Release,2016,2019,...,2015-04-22,NaN,NaN,NaN,NaN,1.0,High,NaN,NaN,NaN
1,NaN,NaN,NaN,42,White,Male,Mount Pleasant Correctional Facility,Prison,2016,2019,...,2015-05-01,NaN,NaN,NaN,NaN,3.0,Low,NaN,NaN,NaN
2,NaN,NaN,NaN,38,White,Female,Iowa Correctional Institution for Women,Work Release,2016,2019,...,2014-09-19,NaN,NaN,NaN,NaN,2.0,Moderate,NaN,NaN,NaN
3,NaN,NaN,NaN,27,Black,Male,Clarinda Correctional Facility,Prison,2016,2019,...,2013-09-30,NaN,NaN,NaN,NaN,2.0,Moderate,NaN,NaN,NaN
4,NaN,NaN,NaN,25,White,Male,Iowa State Penitentiary,Work Release,2016,2019,...,2015-04-01,NaN,NaN,NaN,NaN,3.0,Low,NaN,NaN,NaN


In [3]:
# Step 2: Check the size of the dataset
df.shape

(31703, 37)

In [4]:
# Step 3: Inspect column names, data types, and non-null values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31703 entries, 0 to 31702
Data columns (total 37 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   record_id                        0 non-null      float64
 1   offender_cd                      0 non-null      float64
 2   birth_date                       0 non-null      float64
 3   age                              31703 non-null  int64  
 4   Race                             31703 non-null  object 
 5   sex                              31700 non-null  object 
 6   region_exit                      31703 non-null  object 
 7   exit_sup                         31703 non-null  object 
 8   cohort_fiscal_year               31703 non-null  int64  
 9   report_fiscal_year               31703 non-null  int64  
 10  report_fiscal_year_name          31703 non-null  object 
 11  exit_sup_start_date              31703 non-null  object 
 12  exit_sup_end_date 

In [5]:
# Step 4: Calculate the number and percentage of missing values in each column
missing_values = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_pct": (df.isnull().mean() * 100).round(2)
})

# Step 5: Display only columns containing missing values,
# ordered from highest to lowest percentage missing
missing_values[missing_values["missing_count"] > 0] \
    .sort_values("missing_pct", ascending=False)

,missing_count,missing_pct
record_id,31703,100.00
birth_date,31703,100.00
victimization_risk,31703,100.00
total_victimization_score,31703,100.00
violence_risk,31703,100.00
total_violence_score,31703,100.00
offender_cd,31703,100.00
recid_convicting_crime_type,20004,63.10
recid_convicting_crime_subtype,20004,63.10
recid_convicting_crime_cd_class,20004,63.10


In [6]:
# Step 6: Create a copy of the raw dataset for cleaning
df_clean = df.copy()

In [7]:
# Step 7: Replace missing sex values with "Unknown"
df_clean["sex"] = df_clean["sex"].fillna("Unknown")

# Step 8: Replace missing risk rankings with "Unknown"
df_clean["risk_ranking"] = df_clean["risk_ranking"].fillna("Unknown")

In [8]:
# Step 9: Create age groups for easier comparison
df_clean["age_group"] = pd.cut(
    df_clean["age"],
    bins=[0, 24, 34, 44, 54, float("inf")],
    labels=["Under 25", "25-34", "35-44", "45-54", "55+"]
)

In [9]:
# Step 10: Convert the True/False recidivism field into a numeric 1/0 flag
df_clean["recidivism_flag"] = df_clean["is_recid"].astype(int)

In [10]:
# Step 11: Check the cleaned dataset size and preview the new fields
print("Dataset shape:", df_clean.shape)

df_clean[
    ["age", "age_group", "sex", "risk_ranking", "is_recid", "recidivism_flag"]
].head()

Dataset shape: (31703, 39)


,age,age_group,sex,risk_ranking,is_recid,recidivism_flag
0,25,25-34,Male,High,True,1
1,42,35-44,Male,Low,True,1
2,38,35-44,Female,Moderate,True,1
3,27,25-34,Male,Moderate,True,1
4,25,25-34,Male,Low,True,1


In [11]:
# Step 12: Calculate the overall recidivism rate
total_records = len(df_clean)
recidivism_records = df_clean["recidivism_flag"].sum()
recidivism_rate = df_clean["recidivism_flag"].mean() * 100

print("Total records:", total_records)
print("Recidivism records:", recidivism_records)
print(f"Recidivism rate: {recidivism_rate:.2f}%")

Total records: 31703
Recidivism records: 11699
Recidivism rate: 36.90%


In [12]:
# Step 13: Calculate recidivism statistics by age group
age_analysis = (
    df_clean
    .groupby("age_group", observed=True)
    .agg(
        total_records=("recidivism_flag", "size"),
        recidivism_records=("recidivism_flag", "sum"),
        recidivism_rate=("recidivism_flag", "mean")
    )
    .reset_index()
)

# Convert the recidivism rate to a percentage
age_analysis["recidivism_rate_pct"] = (
    age_analysis["recidivism_rate"] * 100
).round(2)

# Display the results
age_analysis

,age_group,total_records,recidivism_records,recidivism_rate,recidivism_rate_pct
0,Under 25,3285,1336,0.406697,40.67
1,25-34,12123,4839,0.399159,39.92
2,35-44,9323,3411,0.365869,36.59
3,45-54,4756,1609,0.338310,33.83
4,55+,2216,504,0.227437,22.74


In [13]:
# Step 14: Calculate recidivism statistics by risk ranking
risk_analysis = (
    df_clean
    .groupby("risk_ranking")
    .agg(
        total_records=("recidivism_flag", "size"),
        recidivism_records=("recidivism_flag", "sum"),
        recidivism_rate=("recidivism_flag", "mean")
    )
    .reset_index()
)

# Convert the recidivism rate to a percentage
risk_analysis["recidivism_rate_pct"] = (
    risk_analysis["recidivism_rate"] * 100
).round(2)

# Sort from highest to lowest recidivism rate
risk_analysis = risk_analysis.sort_values(
    "recidivism_rate_pct",
    ascending=False
)

# Display the results
risk_analysis

,risk_ranking,total_records,recidivism_records,recidivism_rate,recidivism_rate_pct
0,High,15290,6424,0.420144,42.01
2,Moderate,4776,1699,0.355737,35.57
1,Low,10068,3159,0.313766,31.38
3,Unknown,1569,417,0.265774,26.58


In [16]:
# Step 19: Export the cleaned dataset for Tableau analysis
df_clean.to_csv(
    "../04_clean_data/iowa_recidivism_clean.csv",
    index=False
)